# CUM Series 16 — Per-Head Replication + Blending Combo

Three phases:
- **16a**: Replicate per-head 4-slice result (3 runs)
- **16b**: Combine per-head with TD(λ) + temporal blending recipe
- **16c**: Out_proj column slicing + full wire-up

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
DATA_DIR = 'benchmarks/tier0/data'
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        DATA_PATH)
with open(DATA_PATH, 'r') as f:
    TEXT = f.read()
print(f'Dataset: {len(TEXT):,} chars')

In [ ]:
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum.tensor import PerHeadMuon, PerHeadBlendMuon

MODEL_CFG = dict(d_model=128, n_heads=4, n_layers=4, d_ff=512, ctx_len=256)
BATCH_SIZE = 32
TOTAL_STEPS = 2000
WARMUP_STEPS = 200
EVAL_EVERY = 250
BASE_LR = 0.02
SEED = 42

test_model = MicroGPT(vocab_size=65, **MODEL_CFG)
print(f'Model: {sum(p.numel() for p in test_model.parameters())/1e6:.1f}M params')
del test_model
print('Imports OK')

In [ ]:
class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y

dataset = CharDataset(TEXT, ctx_len=MODEL_CFG['ctx_len'], device=DEVICE)
print(f'Dataset on {DEVICE}: vocab={dataset.vocab_size}')

In [ ]:
class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def train_one(name, optimizers, model, dataset,
              total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE,
              warmup_steps=WARMUP_STEPS, base_lr=BASE_LR, eval_every=EVAL_EVERY):
    model.train()
    trajectory = []

    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        for opt in optimizers:
            opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for step in range(total_steps):
        current_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        for opt in optimizers:
            for g in opt.param_groups:
                if isinstance(opt, torch.optim.AdamW):
                    g['lr'] = get_lr(step, total_steps, warmup_steps, 3e-4)
                else:
                    g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        for opt in optimizers:
            opt.zero_grad()
        loss.backward()
        for opt in optimizers:
            opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
def split_params(model, n_slices=1, col_slices=1, all_sliced=False):
    """Split model params into groups for PerHeadBlendMuon."""
    raw = model._orig_mod if hasattr(model, '_orig_mod') else model
    qkv_params = []
    outproj_params = []
    mlp_up_params = []
    mlp_down_params = []
    other_hidden = []
    other_params = []

    for block in raw.blocks:
        for name, p in block.named_parameters():
            if p.ndim == 2:
                if 'qkv' in name:
                    qkv_params.append(p)
                elif 'out_proj' in name:
                    outproj_params.append(p)
                elif 'up' in name:
                    mlp_up_params.append(p)
                elif 'down' in name:
                    mlp_down_params.append(p)
                else:
                    other_hidden.append(p)
            else:
                other_params.append(p)

    hidden_set = set(id(p) for block in raw.blocks for p in block.parameters())
    for p in raw.parameters():
        if id(p) not in hidden_set:
            other_params.append(p)

    groups = []
    # QKV: row-slice by head
    if qkv_params:
        groups.append({'params': qkv_params, 'n_slices': n_slices, 'col_slices': 1})
    # Out proj: col-slice by head
    if outproj_params:
        groups.append({'params': outproj_params, 'n_slices': 1, 'col_slices': col_slices})
    # MLP: optionally slice
    if all_sliced and n_slices > 1:
        if mlp_up_params:
            groups.append({'params': mlp_up_params, 'n_slices': n_slices, 'col_slices': 1})
        if mlp_down_params:
            groups.append({'params': mlp_down_params, 'n_slices': 1, 'col_slices': n_slices})
    else:
        remaining = mlp_up_params + mlp_down_params
        if remaining:
            groups.append({'params': remaining, 'n_slices': 1, 'col_slices': 1})
    # Other hidden 2D
    if other_hidden:
        groups.append({'params': other_hidden, 'n_slices': 1, 'col_slices': 1})

    return groups, other_params


def make_model_and_opts(dataset, cfg):
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **MODEL_CFG).to(DEVICE)
    model = torch.compile(model, mode='reduce-overhead')
    raw = model._orig_mod if hasattr(model, '_orig_mod') else model

    t = cfg['type']

    if t == 'Muon+AdamW':
        hidden_2d = raw.get_hidden_2d_params()
        other = raw.get_other_params()
        muon = SimpleMuon(hidden_2d, lr=BASE_LR, ns_steps=5)
        adam = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)
        optimizers = [muon, adam]

    elif t == 'PerHead':
        n_slices = cfg.get('n_slices', 4)
        qkv_params = []
        other_hidden = []
        other_params = []
        for block in raw.blocks:
            for name, p in block.named_parameters():
                if p.ndim == 2:
                    if 'qkv' in name:
                        qkv_params.append(p)
                    else:
                        other_hidden.append(p)
                else:
                    other_params.append(p)
        hidden_set = set(id(p) for block in raw.blocks for p in block.parameters())
        for p in raw.parameters():
            if id(p) not in hidden_set:
                other_params.append(p)
        param_groups = [
            {'params': qkv_params, 'n_slices': n_slices},
            {'params': other_hidden, 'n_slices': 1},
        ]
        ph = PerHeadMuon(param_groups, lr=BASE_LR, ns_steps=5)
        adam = torch.optim.AdamW(other_params, lr=3e-4, weight_decay=0.01)
        optimizers = [ph, adam]

    elif t == 'PerHeadBlend':
        n_slices = cfg.get('n_slices', 4)
        col_slices = cfg.get('col_slices', 1)
        mode = cfg.get('mode', 'plain')
        deriv = cfg.get('deriv', -1.0)
        all_sliced = cfg.get('all_sliced', False)

        groups, other_params = split_params(model, n_slices, col_slices, all_sliced)
        ph = PerHeadBlendMuon(
            groups, lr=BASE_LR, ns_steps=5,
            mode=mode, deriv=deriv,
            td_lambda=0.5, input_blend_beta=0.5, input_blend_alpha=0.15,
        )
        adam = torch.optim.AdamW(other_params, lr=3e-4, weight_decay=0.01)
        optimizers = [ph, adam]

    else:
        raise ValueError(f'Unknown: {t}')

    return model, optimizers


def run_all(configs):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, optimizers = make_model_and_opts(dataset, cfg)
            val_loss, traj, elapsed = train_one(name, optimizers, model, dataset)
            results.append(dict(name=name, type=cfg['type'], val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, type=cfg['type'], val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


def show_results(results, series='16'):
    muon = next((r['val_loss'] for r in results if r['type'] == 'Muon+AdamW'), None)
    print(f'\n## Series {series} Results\n')
    print(f'| Config | Val Loss | vs Muon+AdamW | Time |')
    print(f'|--------|----------|---------------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r.get('error'):
            print(f'| {r["name"]} | FAILED | -- | {r["error"][:30]} |')
            continue
        vm = f'{r["val_loss"] - muon:+.4f}' if muon else '--'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vm} | {r["elapsed"]:.0f}s |')


print('Factory ready')

---
## 16a: Replication — Per-Head 4 Slices (3 runs)

Confirm that -0.025 vs Muon is real and not noise.

In [ ]:
CONFIGS_16A = [
    {'name': 'Muon+AdamW', 'type': 'Muon+AdamW'},
    {'name': 'PerHead 4s run1', 'type': 'PerHead', 'n_slices': 4},
    {'name': 'PerHead 4s run2', 'type': 'PerHead', 'n_slices': 4},
    {'name': 'PerHead 4s run3', 'type': 'PerHead', 'n_slices': 4},
]

results_16a = run_all(CONFIGS_16A)
show_results(results_16a, '16a: Replication')

---
## 16b: Per-Head + Blending Recipe

Combine per-head 4-slice with the TD(λ)+temporal blending from Series 12.
If effects are orthogonal: -0.025 (per-head) + -0.009 (blending) = -0.034 vs Muon.

In [ ]:
CONFIGS_16B = [
    {'name': 'Muon+AdamW', 'type': 'Muon+AdamW'},
    {'name': 'PerHead 4s plain', 'type': 'PerHeadBlend', 'n_slices': 4, 'mode': 'plain'},
    {'name': 'PerHead 4s combined', 'type': 'PerHeadBlend', 'n_slices': 4, 'mode': 'combined'},
    {'name': 'PerHead 4s td d=-1.0', 'type': 'PerHeadBlend', 'n_slices': 4, 'mode': 'td', 'deriv': -1.0},
]

results_16b = run_all(CONFIGS_16B)
show_results(results_16b, '16b: Per-Head + Blending')

---
## 16c: Out_Proj Column Slicing + Full Wire-Up

out_proj is (128, 128). Columns 0-31 = head 0 output, 32-63 = head 1, etc.
col_slices=4 transposes, splits by head, NS each, recombines.
Also test slicing MLP weights (all_sliced=True).

In [ ]:
CONFIGS_16C = [
    {'name': 'Muon+AdamW', 'type': 'Muon+AdamW'},
    {'name': 'PerHead QKV td', 'type': 'PerHeadBlend', 'n_slices': 4, 'col_slices': 1, 'mode': 'td', 'deriv': -1.0},
    {'name': 'PerHead QKV+outproj td', 'type': 'PerHeadBlend', 'n_slices': 4, 'col_slices': 4, 'mode': 'td', 'deriv': -1.0},
    {'name': 'Full wire td', 'type': 'PerHeadBlend', 'n_slices': 4, 'col_slices': 4, 'mode': 'td', 'deriv': -1.0, 'all_sliced': True},
]

results_16c = run_all(CONFIGS_16C)
show_results(results_16c, '16c: Out_Proj + Full Wire-Up')